# Feature Engineering

This notebook focuses on transforming telemetry data into actionable performance indicators. By calculating metrics for speed, braking, throttle application, and gear usage, the process creates a structured dataset that quantifies driving behavior, enabling a deeper understanding of track management.

##  Team and Grand Prix Selection
Define parameters for the current analysis, including the team, Grand Prix, and racing season. This scope ensures that the feature engineering process targets the correct, previously processed dataset.

In [1]:
team = "McLaren"
grand_prix = "Japanese Grand Prix"
year = 2025

## Setup and Dependencies

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Fetch cleaned and interpolated telemetry data for both teammates. Loading pre-prepared data ensures that feature calculations are performed on standardized, high-quality inputs.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(year, grand_prix, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (205.96 MB) C:\Users\joseg\AppData\Local\Temp\fastf1


core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix NOR and PIA found in cache.
Driver 1: NOR, Driver 2: PIA, Segment: Q3


## Compute Speed Metrics

Compute speed-related performance metrics such as: entry speed, minimum speed and exit speed throughout the corners. This analysis examines speed profiles relative to track turns to identify differences in corner entry and exit speeds, providing insights into raw pace and efficiency.

In [4]:
try:
    speed_metrics_1 = analyze_speed_metrics(lap_1["Telemetry"], turns)
    speed_metrics_2 = analyze_speed_metrics(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Speed metrics analysis failed: {e}")

print(f"Speed metrics for {driver_1}:\n\n{speed_metrics_1}\n")
print(f"Speed metrics for {driver_2}:\n\n{speed_metrics_2}")

Speed metrics for NOR:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  311.245121    218.368752  213.405535
1      2  213.405535    184.000000  197.114315
2      3  241.647429    244.335719  255.561676
3      4  256.453098    233.000000  233.000000
4      5  241.603092    228.062850  230.621421
5      6  233.273759    205.085156  210.782589
6      7  248.653695    249.708642  266.086864
7      8  291.724639    259.256992  260.903553
8      9  237.589045    149.445000  157.109399
9     10  259.170771    258.521998  246.410158
10    11  156.459127     74.340971  114.403972
11    12  227.128361    230.175784  256.759484
12    13  286.202855    244.035043  242.764766
13    14  211.498151    185.030940  203.221631
14    15  300.618513    297.000000  296.717237
15    16  131.556080     95.997954  115.203419
16    17  102.000000    103.048691  202.393038
17    18  225.438561    228.875612  255.655960

Speed metrics for PIA:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  3

## Compute Braking Zones

Identify and analyze braking zones throughout the lap. This section evaluates how each driver approaches braking points—specifically focusing on the start and end of braking events—to highlight differences in commitment and corner entry style registering information such as: braking duration, braking distance and braking point.

In [5]:
try:
    braking_zones_1 = analyze_braking_zones(lap_1["Telemetry"], turns)
    braking_zones_2 = analyze_braking_zones(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Braking zones analysis failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

Braking zones for NOR:

:     Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0      1     697.701122   803.681039         1.656893       105.979917   
1      2       0.000000     0.000000         0.000000         0.000000   
2      3       0.000000     0.000000         0.000000         0.000000   
3      4       0.000000     0.000000         0.000000         0.000000   
4      5       0.000000     0.000000         0.000000         0.000000   
5      6    1457.223863  1474.887182         0.280348        17.663320   
6      7       0.000000     0.000000         0.000000         0.000000   
7      8       0.000000     0.000000         0.000000         0.000000   
8      9    2349.221500  2411.043118         1.189932        61.821618   
9     10       0.000000     0.000000         0.000000         0.000000   
10    11    2790.804489  2879.121086         2.267196        88.316598   
11    12       0.000000     0.000000         0.000000         0.000000   
12    13    

## Compute Throttle Segments

Calculate throttle usage segments. This analysis determines when and how aggressively each driver applies the throttle coming out of corners, serving as a critical indicator of traction management and power deployment. Each segment registers: mean throttle, throttle standard deviation and full throttle percentage. 

In [6]:
try:
    throttle_segments_1 = analyze_throttle_segments(lap_1["Telemetry"], turns)
    throttle_segments_2 = analyze_throttle_segments(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Throttle segments analysis failed: {e}")


print(f"Throttle segments for {driver_1}:\n\n: {throttle_segments_1}\n")
print(f"Throttle segments for {driver_2}:\n\n: {throttle_segments_2}\n")

Throttle segments for NOR:

:     Turn  MeanThrottle  StdThrottle  FullThrottlePercentage
0      0     93.186279    21.949716               92.682927
1      1      3.844858     8.210226                0.000000
2      2     93.704258    14.254395               84.615385
3      3     76.094200    32.435066               53.333333
4      4     70.761796    21.536353               26.315789
5      5     45.558102    38.635508               11.111111
6      6     94.106365    12.917103               80.555556
7      7     89.475574    28.108604               87.500000
8      8     30.406947    35.129040                5.882353
9      9     89.873642    25.211773               83.783784
10    10     26.133826    43.246295               23.529412
11    11     87.775974    23.428567               75.862069
12    12     87.115532    30.007062               84.722222
13    13     29.954553    31.001811                0.000000
14    14     98.829084     5.305614               96.666667
15    15  

## Compute Gear Shifts

Examine gear shift patterns across the lap registering: number of shifts, lowest gear, highest gear, minimum rpm, maximum rpm, mean rpn and rpm standard deviation. Comparing gear selection at different track segments reveals variations in driving technique, engine management, and how torque and traction are balanced.

In [7]:
try:
    gear_shifts_1 = analyze_gear_shifts(lap_1["Telemetry"], turns)
    gear_shifts_2 = analyze_gear_shifts(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Gear shifts analysis failed: {e}")

print(f"Gear shifts for {driver_1}:\n\n: {gear_shifts_1}\n")
print(f"Gear shifts for {driver_2}:\n\n: {gear_shifts_2}\n")

Gear shifts for NOR:

:     Turn  NumberOfShifts  LowestGear  HighestGear  MinimumRPM  MaximumRPM  \
0      0               1           7            8       10487       11674   
1      1               3           4            8       10861       12375   
2      2               2           4            6       10891       11929   
3      3               0           6            6       10993       11721   
4      4               0           6            6       10602       10986   
5      5               1           5            6       10479       11544   
6      6               2           5            7       10312       11724   
7      7               0           7            7       10327       12373   
8      8               2           5            6        9343       12375   
9      9               3           4            6        9052       12145   
10    10               3           2            6        6792       12103   
11    11               4           2            6   

## Cache Processed Data

Integrate the newly calculated metrics into the driver data structures and save the augmented dataset to the cache. This ensures that extracted features are persistent and readily available for subsequent performance comparisons and visualization steps.

In [8]:
lap_1["SpeedMetrics"] = speed_metrics_1
lap_1["BrakingZones"] = braking_zones_1 
lap_1["ThrottleSegments"] = throttle_segments_1
lap_1["GearShifts"] = gear_shifts_1


lap_2["SpeedMetrics"] = speed_metrics_2
lap_2["BrakingZones"] = braking_zones_2
lap_2["ThrottleSegments"] = throttle_segments_2
lap_2["GearShifts"] = gear_shifts_2

try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q3_NOR.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q3_NOR.pkl
Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q3_PIA.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q3_PIA.pkl

Processed data for 2025 Japanese Grand Prix saved to cache.
Driver 1: NOR, Driver 2: PIA, Segment: Q3
